# Step 3 — Preprocessing Pipeline

**Goal:** Convert raw audio clips from any supported dataset into fixed-length log-mel spectrogram windows saved as numpy arrays, ready for training.

**Output files written to `data/processed/`:**

| File | Shape | Contents |
|---|---|---|
| `windows.npy` | `(N, 128, 32)` float32 | One log-mel window per row |
| `labels.npy` | `(N,)` int8 | 1 = drone, 0 = non-drone |
| `meta.csv` | N rows | recording_id, dataset, window_idx |
| `splits.json` | — | train / val / test window indices |

**Three dataset adapter types — all produce the same interface:**

| Type | Datasets | Storage |
|---|---|---|
| A — HuggingFace | ahlab/DroneAudioSet, geronimobasso/DADS | Parquet via `load_dataset()` |
| B — Folder WAV | saraalemadi, DataSEC | WAV files, label = folder name |
| C — CSV + WAV | ESC-50, FSD50K, MAD | WAV files + label CSV |

## Imports

In [ ]:
import json
import logging
import pathlib
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from datasets import load_dataset

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger('pipeline')
log.info('Imports and logging OK')

## Constants

- `WINDOW_SEC = 1.0` — training window length; matches YAMNet input and literature standard.
- `HOP_LENGTH = 512` at 16 kHz → 32 time frames per window; matches Stage 2 spectrograms.
- `FMIN = 50 Hz` — captures blade-pass fundamentals (start ~100 Hz) with headroom.
- `SUBSET_MODE = True` — fast testing; set `False` to process the full dataset.

In [ ]:
TARGET_SR       = 16_000
WINDOW_SEC      = 1.0
N_MELS          = 128
N_FFT           = 1024
HOP_LENGTH      = 512
FMIN            = 50
FMAX            = 8_000
DRONE_LABEL     = 1
NON_DRONE_LABEL = 0
SUBSET_MODE     = True   # True = fast test (SUBSET_CLIPS per config); False = full dataset
SUBSET_CLIPS    = 30

WINDOW_SAMPLES = int(WINDOW_SEC * TARGET_SR)
N_FRAMES       = 1 + WINDOW_SAMPLES // HOP_LENGTH   # librosa center=True: pads n_fft//2 each side

log.info(f'Window: {WINDOW_SEC}s = {WINDOW_SAMPLES} samples = {N_FRAMES} time frames')
log.info(f'Feature shape per window: ({N_MELS}, {N_FRAMES})')
log.info(f'Subset mode: {SUBSET_MODE}  ({SUBSET_CLIPS} clips/config)')

## Output paths

In [ ]:
_cwd = pathlib.Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd

BASE_DIR     = PROJECT_ROOT / 'data' / 'processed'
WINDOWS_PATH = BASE_DIR / 'windows.npy'
LABELS_PATH  = BASE_DIR / 'labels.npy'
META_PATH    = BASE_DIR / 'meta.csv'
SPLITS_PATH  = BASE_DIR / 'splits.json'

BASE_DIR.mkdir(parents=True, exist_ok=True)

for p in [WINDOWS_PATH, LABELS_PATH, META_PATH, SPLITS_PATH]:
    log.info(f'  {p.name}: {"exists" if p.exists() else "will be created"}')
log.info(f'Output root: {BASE_DIR}')

## Core pipeline functions

Each raw audio clip passes through 4 steps:

```
audio_array
  → to_mono()             (samples, channels) or (samples,) → (samples,) float32
  → resample_if_needed()  any Hz → 16 000 Hz
  → segment_audio()       long clip → list of 1-second windows
  → compute_log_mel()     each window → (128, 32) float32
```

Each function has its own definition cell and its own test cell so failures are pinpointed immediately.

In [ ]:
def to_mono(y: np.ndarray) -> np.ndarray:
    """Take channel 0 if multi-channel; cast to float32."""
    y = np.asarray(y, dtype=np.float32)
    if y.ndim == 2:
        y = y[:, 0]
    return y

log.info('to_mono: defined')

In [ ]:
_multi = np.ones((100, 8), dtype=np.float64)
assert to_mono(_multi).shape == (100,),  f'expected (100,), got {to_mono(_multi).shape}'
assert to_mono(_multi).dtype == np.float32

_mono = np.zeros(50, dtype=np.float32)
assert to_mono(_mono).shape == (50,)

log.info('to_mono: tests passed')

In [ ]:
def resample_if_needed(y: np.ndarray, sr: int) -> np.ndarray:
    """Resample y to TARGET_SR if sr != TARGET_SR. Returns float32."""
    if sr == TARGET_SR:
        return y
    return librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)

log.info('resample_if_needed: defined')

In [ ]:
_y44 = np.random.randn(44_100).astype(np.float32)
_y16 = resample_if_needed(_y44, 44_100)
assert len(_y16) == TARGET_SR, f'expected {TARGET_SR} samples, got {len(_y16)}'

_y_same = np.zeros(TARGET_SR, dtype=np.float32)
assert resample_if_needed(_y_same, TARGET_SR) is _y_same   # no copy when SR already correct

log.info(f'resample_if_needed: tests passed  (44100 samples @ 44.1kHz → {len(_y16)} samples @ 16kHz)')

In [ ]:
def segment_audio(y: np.ndarray) -> list:
    """Split y (at TARGET_SR) into non-overlapping WINDOW_SAMPLES chunks. Drops trailing remainder."""
    n_windows = len(y) // WINDOW_SAMPLES
    return [y[i * WINDOW_SAMPLES : (i + 1) * WINDOW_SAMPLES] for i in range(n_windows)]

log.info('segment_audio: defined')

In [ ]:
_y90 = np.zeros(TARGET_SR * 90, dtype=np.float32)
assert len(segment_audio(_y90)) == 90
assert len(segment_audio(_y90)[0]) == WINDOW_SAMPLES

_y_short = np.zeros(TARGET_SR // 2, dtype=np.float32)
assert len(segment_audio(_y_short)) == 0   # 0.5 s clip has no full windows

log.info('segment_audio: tests passed  (90s → 90 windows; 0.5s → 0 windows)')

In [ ]:
def compute_log_mel(window: np.ndarray) -> np.ndarray:
    """Compute log-mel spectrogram for one WINDOW_SAMPLES clip. Returns (N_MELS, N_FRAMES) float32."""
    S = librosa.feature.melspectrogram(
        y=window, sr=TARGET_SR,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    )
    return np.log1p(S).astype(np.float32)

log.info('compute_log_mel: defined')

In [ ]:
_win  = np.random.randn(WINDOW_SAMPLES).astype(np.float32)
_feat = compute_log_mel(_win)

assert _feat.shape == (N_MELS, N_FRAMES), f'expected ({N_MELS}, {N_FRAMES}), got {_feat.shape}'
assert _feat.dtype == np.float32
assert np.all(_feat >= 0),          'log1p output must be non-negative'
assert not np.isnan(_feat).any(),   'NaN in log-mel output'

log.info(f'compute_log_mel: tests passed  shape={_feat.shape}  range=[{_feat.min():.3f}, {_feat.max():.3f}]')

In [ ]:
def process_clip(
    audio_array: np.ndarray,
    sr: int,
    label: int,
    recording_id: str,
    dataset_name: str,
) -> list:
    """Run the full 4-step pipeline on one raw audio clip. Returns one dict per window."""
    y = to_mono(audio_array)
    y = resample_if_needed(y, sr)
    windows = segment_audio(y)
    return [
        {
            'window':       compute_log_mel(w),
            'label':        label,
            'recording_id': recording_id,
            'dataset':      dataset_name,
            'window_idx':   i,
        }
        for i, w in enumerate(windows)
    ]

log.info('process_clip: defined')

### Dry-run — synthetic drone signal

Verify the full pipeline end-to-end before touching real data. Uses a 200 Hz tone with harmonics (blade-pass frequency signature at 200, 400, 600 Hz).

In [ ]:
_t = np.linspace(0, 5, TARGET_SR * 5, dtype=np.float32)
_sig = (
    0.5 * np.sin(2 * np.pi * 200 * _t)
    + 0.3 * np.sin(2 * np.pi * 400 * _t)
    + 0.1 * np.sin(2 * np.pi * 600 * _t)
)

_results = process_clip(_sig, TARGET_SR, DRONE_LABEL, 'synth_001', 'synthetic')

assert len(_results) == 5,                                  f'expected 5 windows, got {len(_results)}'
assert _results[0]['window'].shape == (N_MELS, N_FRAMES)
assert _results[0]['label']         == DRONE_LABEL
assert _results[0]['recording_id']  == 'synth_001'
assert _results[4]['window_idx']    == 4

log.info(f'Dry-run: 5s synthetic clip → {len(_results)} windows, shape={_results[0]["window"].shape}')

In [ ]:
def run_pipeline(
    clip_iter,
    dataset_name: str,
    windows_list: list,
    labels_list: list,
    meta_list: list,
    log_every: int = 5,
) -> None:
    """Iterate clip_iter, run process_clip on each clip, append results to accumulators."""
    n_clips = 0
    n_windows = 0
    for audio_array, sr, label, recording_id in clip_iter:
        for r in process_clip(audio_array, sr, label, recording_id, dataset_name):
            windows_list.append(r['window'])
            labels_list.append(r['label'])
            meta_list.append({
                'recording_id': r['recording_id'],
                'dataset':      r['dataset'],
                'window_idx':   r['window_idx'],
            })
            n_windows += 1
        n_clips += 1
        if n_clips % log_every == 0:
            log.info(f'  [{dataset_name}] {n_clips} clips processed, {n_windows} windows')
    log.info(f'  [{dataset_name}] Done: {n_clips} clips → {n_windows} windows')

log.info('run_pipeline: defined')

## Dataset adapters

All three types yield the same tuple: `(audio_array, sample_rate, label, recording_id)`.  
After this point, `run_pipeline` handles everything identically regardless of source.

| Type | When to use | Label source |
|---|---|---|
| A — HuggingFace | ahlab, geronimobasso | Passed via `label_fn` callable |
| B — Folder WAV | saraalemadi, DataSEC | Substring of folder path |
| C — CSV + WAV | ESC-50, FSD50K, MAD | CSV column value |

### Type A — HuggingFace

ahlab splits are named `train_001`, `train_002`, ... The adapter streams them one by one, stopping when `n_clips` is reached or a split raises an exception (no more splits).

- **ahlab:** label known from which config we load → `label_fn = lambda _: DRONE_LABEL`
- **geronimobasso:** label in `item['label']` → `label_fn = lambda item: int(item['label'])`

In [ ]:
def iter_huggingface_dataset(
    repo_id: str,
    config: str,
    label_fn,
    path_field: str = 'file_path',
    n_clips: int = None,
):
    """Yield (audio_array, sr, label, recording_id) from a HuggingFace dataset."""
    collected = 0
    split_idx = 1
    while n_clips is None or collected < n_clips:
        split_name = f'train_{split_idx:03d}'
        try:
            stream = load_dataset(repo_id, config, streaming=True, split=split_name)
            for item in stream:
                if n_clips is not None and collected >= n_clips:
                    return
                audio = item['audio']
                recording_id = item.get(path_field, f'{config}_{split_name}_{collected}')
                yield np.array(audio['array']), audio['sampling_rate'], label_fn(item), recording_id
                collected += 1
        except Exception:
            log.info(f'  {repo_id}/{config}: no more splits after train_{split_idx - 1:03d}')
            return
        split_idx += 1

log.info('iter_huggingface_dataset: defined')

### Type B — Folder WAV

Label is encoded in the folder path. Pass a `label_map` dict where keys are substrings matched case-insensitively against the full file path.

**saraalemadi example:** `label_map = {'drone': 1, 'unknown': 0, 'silence': 0}`  
**DataSEC example (hard negatives only):** `label_map = {'helicopter': 0, 'propeller': 0}`

In [ ]:
def iter_folder_dataset(root, label_map: dict, dataset_name: str):
    """Yield (audio_array, sr, label, recording_id) by walking a directory tree."""
    root = pathlib.Path(root)
    matched = skipped = 0
    for wav_path in sorted(root.rglob('*.wav')):
        path_lower = str(wav_path).lower()
        label = next(
            (lbl for key, lbl in label_map.items() if key.lower() in path_lower),
            None,
        )
        if label is None:
            skipped += 1
            continue
        try:
            y, sr = sf.read(str(wav_path), always_2d=True)
            yield y, sr, label, str(wav_path.relative_to(root))
            matched += 1
        except Exception as e:
            log.warning(f'  Failed to load {wav_path.name}: {e}')
            skipped += 1
    log.info(f'iter_folder_dataset ({dataset_name}): {matched} yielded, {skipped} skipped')

log.info('iter_folder_dataset: defined')

### Type C — CSV + WAV

The CSV maps filenames to labels. Pass a `label_map` dict translating raw CSV values to 0/1. Rows not in `label_map` are skipped — useful for selecting only relevant classes from large multi-class datasets.

**ESC-50 example (helicopter only):** `label_map = {'helicopter': 0}`  
**MAD example:** `label_map = {'helicopter': 0, 'fighter': 0}`

In [ ]:
def iter_csv_dataset(
    audio_dir,
    csv_path,
    filename_col: str,
    label_col: str,
    label_map: dict,
    dataset_name: str,
):
    """Yield (audio_array, sr, label, recording_id) using a CSV for label lookup."""
    audio_dir = pathlib.Path(audio_dir)
    df = pd.read_csv(csv_path)
    matched = skipped = 0
    for _, row in df.iterrows():
        raw_label = row[label_col]
        if raw_label not in label_map:
            skipped += 1
            continue
        wav_path = audio_dir / row[filename_col]
        if not wav_path.exists():
            log.warning(f'  Missing: {wav_path.name}')
            skipped += 1
            continue
        try:
            y, sr = sf.read(str(wav_path), always_2d=True)
            yield y, sr, label_map[raw_label], row[filename_col]
            matched += 1
        except Exception as e:
            log.warning(f'  Failed to load {wav_path.name}: {e}')
            skipped += 1
    log.info(f'iter_csv_dataset ({dataset_name}): {matched} yielded, {skipped} skipped')

log.info('iter_csv_dataset: defined')

## Process ahlab/DroneAudioSet

| Config | Label | Rationale |
|---|---|---|
| `drone-with-source` | 1 (drone) | Drone + real background — closest to deployment conditions |
| `drone-only` | 1 (drone) | Isolated drone signal — teaches pure acoustic signature |
| `source-only` | 0 (non-drone) | Background only — negative class |

Set `SUBSET_MODE = False` in the constants cell to process all available clips.

In [ ]:
n_clips: int = SUBSET_CLIPS if SUBSET_MODE else None

all_windows: list = []
all_labels:  list = []
all_meta:    list = []

log.info(f'Accumulators initialized  |  n_clips_per_config={n_clips}')

In [ ]:
log.info('Processing drone-with-source (positive class)...')
_iter = iter_huggingface_dataset(
    'ahlab-drone-project/DroneAudioSet',
    'drone-with-source',
    label_fn=lambda _: DRONE_LABEL,
    n_clips=n_clips,
)
run_pipeline(_iter, 'ahlab-drone-with-source', all_windows, all_labels, all_meta)
log.info(f'Running total: {len(all_windows)} windows')

In [ ]:
log.info('Processing drone-only (positive class)...')
_iter = iter_huggingface_dataset(
    'ahlab-drone-project/DroneAudioSet',
    'drone-only',
    label_fn=lambda _: DRONE_LABEL,
    n_clips=n_clips,
)
run_pipeline(_iter, 'ahlab-drone-only', all_windows, all_labels, all_meta)
log.info(f'Running total: {len(all_windows)} windows')

In [ ]:
log.info('Processing source-only (negative class)...')
_iter = iter_huggingface_dataset(
    'ahlab-drone-project/DroneAudioSet',
    'source-only',
    label_fn=lambda _: NON_DRONE_LABEL,
    n_clips=n_clips,
)
run_pipeline(_iter, 'ahlab-source-only', all_windows, all_labels, all_meta)
log.info(f'Running total: {len(all_windows)} windows')

## Stack arrays and check class distribution

In [ ]:
X       = np.stack(all_windows, axis=0)
y       = np.array(all_labels, dtype=np.int8)
meta_df = pd.DataFrame(all_meta)

n_drone    = int(y.sum())
n_nondrone = int((y == 0).sum())
ratio      = n_drone / max(n_nondrone, 1)

log.info(f'X shape: {X.shape}  dtype={X.dtype}')
log.info(f'y shape: {y.shape}  dtype={y.dtype}')
log.info(f'Drone windows:     {n_drone}')
log.info(f'Non-drone windows: {n_nondrone}')
log.info(f'Imbalance ratio:   {ratio:.2f}:1  {"(balanced)" if ratio < 3 else "consider class weights"}')

## Save to disk

In [ ]:
np.save(WINDOWS_PATH, X)
np.save(LABELS_PATH, y)
meta_df.to_csv(META_PATH, index=False)

log.info(f'Saved {WINDOWS_PATH.name}   ({WINDOWS_PATH.stat().st_size / 1e6:.1f} MB)')
log.info(f'Saved {LABELS_PATH.name}    ({LABELS_PATH.stat().st_size / 1e6:.2f} MB)')
log.info(f'Saved {META_PATH.name}      ({META_PATH.stat().st_size / 1e3:.1f} kB)')

## Adding future datasets

Uncomment the relevant block once the dataset folder is downloaded. Re-run the stack and save cells after adding new data.

In [ ]:
# ── saraalemadi/DroneAudioDataset  (Type B — folder WAV) ──────────────────
# _iter = iter_folder_dataset(
#     PROJECT_ROOT / 'data' / 'raw' / 'saraalemadi',
#     label_map={'drone': DRONE_LABEL, 'unknown': NON_DRONE_LABEL, 'silence': NON_DRONE_LABEL},
#     dataset_name='saraalemadi',
# )
# run_pipeline(_iter, 'saraalemadi', all_windows, all_labels, all_meta)

# ── ESC-50  (Type C — helicopter class as hard negative) ──────────────────
# _iter = iter_csv_dataset(
#     PROJECT_ROOT / 'data' / 'raw' / 'ESC-50' / 'audio',
#     PROJECT_ROOT / 'data' / 'raw' / 'ESC-50' / 'meta' / 'esc50.csv',
#     filename_col='filename', label_col='category',
#     label_map={'helicopter': NON_DRONE_LABEL},
#     dataset_name='ESC-50-helicopter',
# )
# run_pipeline(_iter, 'ESC-50', all_windows, all_labels, all_meta)

# ── MAD — Military Audio Dataset  (Type C — helicopter + fighter) ─────────
# _iter = iter_csv_dataset(
#     PROJECT_ROOT / 'data' / 'raw' / 'MAD' / 'data',
#     PROJECT_ROOT / 'data' / 'raw' / 'MAD' / 'annotation.csv',
#     filename_col='fname', label_col='label',
#     label_map={'helicopter': NON_DRONE_LABEL, 'fighter': NON_DRONE_LABEL},
#     dataset_name='MAD',
# )
# run_pipeline(_iter, 'MAD', all_windows, all_labels, all_meta)

# ── geronimobasso/DADS  (Type A — HuggingFace, binary label in item) ──────
# _iter = iter_huggingface_dataset(
#     'geronimobasso/drone-audio-detection-samples',
#     config='default',
#     label_fn=lambda item: int(item['label']),
#     path_field='audio',
#     n_clips=None,
# )
# run_pipeline(_iter, 'geronimobasso', all_windows, all_labels, all_meta)

log.info('Future dataset placeholders ready')

## Train / validation / test split

**Why split at the recording level, not the window level:**

One 90-second clip produces ~90 one-second windows. A random window-level split places windows from the same recording in both train and test. The model then sees near-identical audio during training and evaluation — inflating test accuracy and hiding real generalisation failures.

Splitting by recording guarantees every window from a given recording lives in exactly one partition.

**Ratios:** 80 % train / 10 % val / 10 % test

In [ ]:
rng = np.random.default_rng(42)
unique_recs = meta_df['recording_id'].unique()
rng.shuffle(unique_recs)

n = len(unique_recs)
train_recs = set(unique_recs[:int(n * 0.80)])
val_recs   = set(unique_recs[int(n * 0.80):int(n * 0.90)])
test_recs  = set(unique_recs[int(n * 0.90):])

log.info(f'Unique recordings: {n}')
log.info(f'  Train: {len(train_recs)}  Val: {len(val_recs)}  Test: {len(test_recs)}')

In [ ]:
rec_arr   = meta_df['recording_id'].values
train_idx = np.where([r in train_recs for r in rec_arr])[0]
val_idx   = np.where([r in val_recs   for r in rec_arr])[0]
test_idx  = np.where([r in test_recs  for r in rec_arr])[0]

assert not (set(rec_arr[train_idx]) & set(rec_arr[val_idx])),  'Recording leak: train ∩ val'
assert not (set(rec_arr[train_idx]) & set(rec_arr[test_idx])), 'Recording leak: train ∩ test'
assert not (set(rec_arr[val_idx])   & set(rec_arr[test_idx])), 'Recording leak: val ∩ test'

log.info(f'Train: {len(train_idx):5d} windows  (drone={y[train_idx].sum()}, non={int((y[train_idx]==0).sum())})')
log.info(f'Val:   {len(val_idx):5d} windows  (drone={y[val_idx].sum()}, non={int((y[val_idx]==0).sum())})')
log.info(f'Test:  {len(test_idx):5d} windows  (drone={y[test_idx].sum()}, non={int((y[test_idx]==0).sum())})')
log.info('No recording leaks across splits ✓')

In [ ]:
splits = {
    'train': train_idx.tolist(),
    'val':   val_idx.tolist(),
    'test':  test_idx.tolist(),
}
with open(SPLITS_PATH, 'w') as f:
    json.dump(splits, f)

log.info(f'Saved {SPLITS_PATH.name}  ({SPLITS_PATH.stat().st_size / 1e3:.1f} kB)')

## Output verification

Load all saved files from disk and verify correctness. Every check here must pass before this notebook is considered complete.

In [ ]:
X_v = np.load(WINDOWS_PATH)
y_v = np.load(LABELS_PATH)
m_v = pd.read_csv(META_PATH)
with open(SPLITS_PATH) as f:
    s_v = json.load(f)

log.info(f'windows.npy : {X_v.shape}  dtype={X_v.dtype}')
log.info(f'labels.npy  : {y_v.shape}  dtype={y_v.dtype}')
log.info(f'meta.csv    : {len(m_v)} rows  columns={list(m_v.columns)}')
log.info(f'splits.json : train={len(s_v["train"])}  val={len(s_v["val"])}  test={len(s_v["test"])}')

In [ ]:
assert X_v.shape[1] == N_MELS,    f'mel bins: expected {N_MELS}, got {X_v.shape[1]}'
assert X_v.shape[2] == N_FRAMES,  f'time frames: expected {N_FRAMES}, got {X_v.shape[2]}'
assert X_v.dtype   == np.float32, f'expected float32, got {X_v.dtype}'
assert y_v.dtype   == np.int8,    f'expected int8, got {y_v.dtype}'
assert set(np.unique(y_v)) <= {0, 1}
assert not np.isnan(X_v).any(),   'NaN found in windows.npy'
assert not np.isinf(X_v).any(),   'Inf found in windows.npy'
assert len(X_v) == len(y_v) == len(m_v), 'Length mismatch between arrays'

log.info(f'Value range : [{X_v.min():.4f}, {X_v.max():.4f}]  mean={X_v.mean():.4f}')
log.info('All assertions passed ✓')

In [ ]:
for split_name, idxs in [('train', s_v['train']), ('val', s_v['val']), ('test', s_v['test'])]:
    y_split = y_v[idxs]
    n_d = int(y_split.sum())
    n_n = int((y_split == 0).sum())
    ratio = n_d / max(n_n, 1)
    warn = '  ⚠ imbalance > 3:1' if ratio > 3 or ratio < 1 / 3 else ''
    log.info(f'{split_name:5s}: {len(idxs):5d} windows  drone={n_d:4d}  non-drone={n_n:4d}  ratio={ratio:.2f}{warn}')

In [ ]:
drone_idxs    = np.where(y_v == DRONE_LABEL)[0][:3]
nondrone_idxs = np.where(y_v == NON_DRONE_LABEL)[0][:3]

fig, axes = plt.subplots(2, 3, figsize=(14, 6))

for col, idx in enumerate(drone_idxs):
    axes[0, col].imshow(X_v[idx], aspect='auto', origin='lower', cmap='inferno')
    axes[0, col].set_title(f'Drone window #{idx}', fontsize=9)

for col, idx in enumerate(nondrone_idxs):
    axes[1, col].imshow(X_v[idx], aspect='auto', origin='lower', cmap='inferno')
    axes[1, col].set_title(f'Non-drone window #{idx}', fontsize=9)

for ax in axes.flat:
    ax.set_xlabel('Time frame')
axes[0, 0].set_ylabel('Mel bin')
axes[1, 0].set_ylabel('Mel bin')

plt.suptitle(
    f'Sample log-mel windows from windows.npy\n'
    f'Shape per window: ({N_MELS}, {N_FRAMES})  |  Drone (top) vs Non-drone (bottom)'
)
plt.tight_layout()
plt.show()
log.info('Visualisation complete')